# Классификация: превышает ли значение SI медианное значение выборки

In [49]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

import tqdm
from tqdm.auto import tqdm

import pickle

In [50]:
RANDOM_STATE = 22

Загружаем данные. Признаки уже стандартизированы, а таргет логарифмирован, поэтому первым преобразованием будет возврат таргета в исходное состояние.

In [51]:
train_df = pd.read_csv("../../data/processed/si_train.csv")
test_df = pd.read_csv("../../data/processed/si_test.csv")

In [52]:
X_train = train_df.drop(columns=["SI"])
X_test = test_df.drop(columns=["SI"])

y_train = 10**(-train_df["SI"])
y_test = 10**(-test_df["SI"])

# Сразу преобразуем наши игреки для классификации
median_y = y_train.median()
y_train = (y_train > median_y).astype(int)

# преобразуем y_test с медианой полученой на train подвыборке
y_test = (y_test > median_y).astype(int)

print(f"В обучающей выборке содержится {X_train.shape[0]} строк и {X_train.shape[1]} признаков")
print(f"В тестовой выборке содержится {X_test.shape[0]} строк и {X_test.shape[1]} признаков")

В обучающей выборке содержится 798 строк и 30 признаков
В тестовой выборке содержится 200 строк и 30 признаков


Зададим параметры сетки для подбора параметров трех разных моделей регрессии.

In [53]:
param_distributions = {
    "SVC": {
        "model": SVC(probability=True, random_state=42),
        "params": {
            "C": [0.1, 1.0, 10.0],
            "gamma": ["scale", "auto", 0.01]
        }
    },
    "RandomForest": {
        'model': RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        'params': {
            'n_estimators': [100, 150, 200],
            'max_depth': [3, 5, 8, 12],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4]
        }
    },
    "LightGBM": {
        "model": LGBMClassifier(random_state=RANDOM_STATE, verbose=-1),
        "params": {"learning_rate": [0.01, 0.1], "n_estimators": [100, 200]}
    }
}

In [54]:
best_models = {}
for name, config in tqdm(param_distributions.items()):
    print(f"\n  Модель: {name}")
    print("="*50)
    
    grid_search = GridSearchCV(
        estimator=config["model"],
        param_grid=config["params"],
        cv=5,
        scoring="roc_auc",
        n_jobs=-1
    )
    
    grid_search.fit(X_train, y_train)
    
    # Оцениваем лучшую модель на тестовой выборке
    best_model = grid_search.best_estimator_
    best_models[name] = best_model
    
    preds = best_model.predict(X_test)
    probs = best_model.predict_proba(X_test)[:, 1]

    print(f"    Лучшие параметры: {grid_search.best_params_}")
    print(f"    Accuracy на тестовой выборке: {accuracy_score(y_test, preds):.4f}")
    print(f"    ROC-AUC на тестовой выборке: {roc_auc_score(y_test, probs):.4f}")

  0%|          | 0/3 [00:00<?, ?it/s]


  Модель: SVC
    Лучшие параметры: {'C': 10.0, 'gamma': 'auto'}
    Accuracy на тестовой выборке: 0.6650
    ROC-AUC на тестовой выборке: 0.7018

  Модель: RandomForest
    Лучшие параметры: {'max_depth': 8, 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 100}
    Accuracy на тестовой выборке: 0.6600
    ROC-AUC на тестовой выборке: 0.7117

  Модель: LightGBM
    Лучшие параметры: {'learning_rate': 0.01, 'n_estimators': 100}
    Accuracy на тестовой выборке: 0.6350
    ROC-AUC на тестовой выборке: 0.6861


Лучшей моделью по метрике ROC-AUC на тестовой выборке (0.7117) показал себя RandomForest с параметрами 

{'max_depth': 8,<br> 'min_samples_leaf': 1,<br> 'min_samples_split': 10,<br> 'n_estimators': 100}

Сериализуем лучшую модель и сохраним в файл.

In [55]:
with open('../../models/model_si_classification.pkl', 'wb') as output:
    pickle.dump(best_models["RandomForest"], output)